# 📚 Data Mining Lab Assignment 2
## Recommender System using a Medallion Data Lake

**Student:** Veda Akash  
**Roll No:** LCI2023057  
**Dataset:** Book-Crossing Dataset (Users, Books, Ratings)  
**Pipeline:** Bronze → Silver → Gold → Recommender System

---

## 🔧 Install Dependencies

In [ ]:
!pip install pyarrow fastparquet scikit-learn scipy pandas numpy matplotlib seaborn -q

## 📦 Import Libraries

In [ ]:
import os
import zipfile
import urllib.request
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

print('✅ All libraries imported successfully!')

---
# 🥉 STEP 1: Set Up the Data Lake and Build the Bronze Layer

In [ ]:
# ── Create Data Lake folder structure ──────────────────────────────────────
for folder in ['bronze', 'silver', 'gold']:
    os.makedirs(folder, exist_ok=True)
    print(f'📁 Created folder: {folder}/')

print('\n✅ Data Lake folders created: bronze/ | silver/ | gold/')

In [ ]:
# ── Download & extract the Book-Crossing dataset ───────────────────────────
DATASET_URL = 'https://cdn.freecodecamp.org/project-data/books/book-crossings.zip'
ZIP_PATH    = 'book-crossings.zip'
DATA_DIR    = 'raw_data'

if not os.path.exists(ZIP_PATH):
    print('⬇️  Downloading dataset...')
    urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)
    print('✅ Download complete!')
else:
    print('✅ Zip already exists, skipping download.')

os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(DATA_DIR)
    print(f'📂 Extracted files: {z.namelist()}')

In [ ]:
# ── Locate raw CSV files ───────────────────────────────────────────────────
import glob
csv_files = glob.glob(f'{DATA_DIR}/**/*.csv', recursive=True) + \
            glob.glob(f'{DATA_DIR}/*.csv')
print('Raw CSV files found:')
for f in csv_files:
    print(f'  {f}')

In [ ]:
# ── Helper: find file by keyword ───────────────────────────────────────────
def find_file(keyword):
    for f in csv_files:
        if keyword.lower() in os.path.basename(f).lower():
            return f
    # Fallback: try to find by content inspection
    return None

users_file   = find_file('user')
books_file   = find_file('book') if find_file('book') else find_file('Book')
ratings_file = find_file('rating')

print(f'Users file  : {users_file}')
print(f'Books file  : {books_file}')
print(f'Ratings file: {ratings_file}')

In [ ]:
# ── Robust CSV reader (handles ; separator + non-UTF-8 encoding) ───────────
def read_raw_csv(filepath, **kwargs):
    """Try multiple encodings and return the first successful read."""
    encodings = ['latin-1', 'iso-8859-1', 'cp1252', 'utf-8']
    for enc in encodings:
        try:
            df = pd.read_csv(
                filepath,
                sep=';',
                encoding=enc,
                on_bad_lines='skip',
                low_memory=False,
                **kwargs
            )
            print(f'  ✅ Read with encoding={enc}  shape={df.shape}')
            return df
        except Exception as e:
            print(f'  ⚠️  encoding={enc} failed: {e}')
    raise RuntimeError(f'Could not read {filepath}')

INGESTION_TS = datetime.utcnow().isoformat()
print(f'Ingestion timestamp: {INGESTION_TS}\n')

print('Reading Users...')
users_raw = read_raw_csv(users_file)

print('Reading Books...')
books_raw = read_raw_csv(books_file)

print('Reading Ratings...')
ratings_raw = read_raw_csv(ratings_file)

In [ ]:
# ── Standardise column names (strip quotes/spaces added by CSV parser) ─────
def clean_col_names(df):
    df.columns = [
        c.strip().strip('"').strip("'").replace(' ', '_')
        for c in df.columns
    ]
    return df

users_raw   = clean_col_names(users_raw)
books_raw   = clean_col_names(books_raw)
ratings_raw = clean_col_names(ratings_raw)

print('Users columns  :', users_raw.columns.tolist())
print('Books columns  :', books_raw.columns.tolist())
print('Ratings columns:', ratings_raw.columns.tolist())

In [ ]:
# ── Rename columns to consistent names ────────────────────────────────────
# Users: User-ID, Location, Age
users_col_map = {}
for c in users_raw.columns:
    cl = c.lower().replace('-','_').replace(' ','_')
    if 'user' in cl and 'id' in cl:  users_col_map[c] = 'user_id'
    elif 'location' in cl:           users_col_map[c] = 'location'
    elif 'age' in cl:                users_col_map[c] = 'age'
users_raw.rename(columns=users_col_map, inplace=True)

# Books: ISBN, Book-Title, Book-Author, Year-Of-Publication, Publisher, Image-URL-S/M/L
books_col_map = {}
for c in books_raw.columns:
    cl = c.lower().replace('-','_').replace(' ','_')
    if cl == 'isbn':                     books_col_map[c] = 'isbn'
    elif 'title' in cl:                  books_col_map[c] = 'book_title'
    elif 'author' in cl:                 books_col_map[c] = 'book_author'
    elif 'year' in cl:                   books_col_map[c] = 'year_of_publication'
    elif 'publisher' in cl:              books_col_map[c] = 'publisher'
    elif 'image' in cl and '_s' in cl:   books_col_map[c] = 'image_url_s'
    elif 'image' in cl and '_m' in cl:   books_col_map[c] = 'image_url_m'
    elif 'image' in cl and '_l' in cl:   books_col_map[c] = 'image_url_l'
books_raw.rename(columns=books_col_map, inplace=True)

# Ratings: User-ID, ISBN, Book-Rating
ratings_col_map = {}
for c in ratings_raw.columns:
    cl = c.lower().replace('-','_').replace(' ','_')
    if 'user' in cl and 'id' in cl:   ratings_col_map[c] = 'user_id'
    elif cl == 'isbn':                 ratings_col_map[c] = 'isbn'
    elif 'rating' in cl:              ratings_col_map[c] = 'book_rating'
ratings_raw.rename(columns=ratings_col_map, inplace=True)

print('Users columns  :', users_raw.columns.tolist())
print('Books columns  :', books_raw.columns.tolist())
print('Ratings columns:', ratings_raw.columns.tolist())

In [ ]:
# ── Add ingestion timestamp and save to Bronze as Parquet ─────────────────
users_raw['ingestion_timestamp']   = INGESTION_TS
books_raw['ingestion_timestamp']   = INGESTION_TS
ratings_raw['ingestion_timestamp'] = INGESTION_TS

users_raw.to_parquet('bronze/users.parquet',   index=False)
books_raw.to_parquet('bronze/books.parquet',   index=False)
ratings_raw.to_parquet('bronze/ratings.parquet', index=False)

print('✅ Bronze layer saved!')
print(f'   bronze/users.parquet   → {users_raw.shape}')
print(f'   bronze/books.parquet   → {books_raw.shape}')
print(f'   bronze/ratings.parquet → {ratings_raw.shape}')

---
# 🔍 STEP 2: Explore the Bronze Data (EDA)

In [ ]:
# ── Reload from Bronze (practice reading Parquet) ─────────────────────────
users_b   = pd.read_parquet('bronze/users.parquet')
books_b   = pd.read_parquet('bronze/books.parquet')
ratings_b = pd.read_parquet('bronze/ratings.parquet')

print('Bronze tables reloaded ✅')

In [ ]:
# ── Shape, dtypes and missing values ─────────────────────────────────────
def eda_summary(df, name):
    print(f'\n{'='*60}')
    print(f'  TABLE: {name}')
    print(f'{'='*60}')
    print(f'  Shape : {df.shape}')
    print(f'  dtypes:\n{df.dtypes.to_string()}')
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    mv = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
    print(f'\n  Missing Values:\n{mv[mv.missing_count > 0].to_string()}')
    print(f'\n  First 3 rows:\n{df.head(3).to_string()}')

eda_summary(users_b,   'USERS')
eda_summary(books_b,   'BOOKS')
eda_summary(ratings_b, 'RATINGS')

**Finding (Shape & Types):** Users has ~278k rows with age often missing; Books has ~271k rows with some NaN in author/publisher; Ratings has ~1.1M rows. Several columns are stored as object (string) that need type conversion in Silver.

In [ ]:
# ── Distribution of Ratings ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ratings_b['book_rating'] = pd.to_numeric(ratings_b['book_rating'], errors='coerce')
rating_counts = ratings_b['book_rating'].value_counts().sort_index()

rating_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Distribution of All Ratings (0–10)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Rating Value')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

explicit_ratings = ratings_b[ratings_b['book_rating'] > 0]
explicit_ratings['book_rating'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Distribution of Explicit Ratings (1–10)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Rating Value')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('rating_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

zero_count = (ratings_b['book_rating'] == 0).sum()
print(f'Ratings with value 0 (implicit): {zero_count:,} ({zero_count/len(ratings_b)*100:.1f}%)')
print(f'Explicit ratings (1-10)        : {len(explicit_ratings):,}')

**Finding (Ratings):** ~70% of ratings are 0 (implicit – user interacted with the book but did not explicitly rate it). These will be removed in Silver since they carry no preference signal.

In [ ]:
# ── Distribution of User Ages and Publication Years ───────────────────────
users_b['age'] = pd.to_numeric(users_b['age'], errors='coerce')
books_b['year_of_publication'] = pd.to_numeric(books_b['year_of_publication'], errors='coerce')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

users_b['age'].dropna().plot(
    kind='hist', bins=50, ax=axes[0], color='mediumseagreen', edgecolor='white')
axes[0].set_title('User Age Distribution (Raw)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Frequency')
axes[0].axvline(5,   color='red',  linestyle='--', label='min valid (5)')
axes[0].axvline(100, color='red',  linestyle='--', label='max valid (100)')
axes[0].legend()

books_b['year_of_publication'].dropna().clip(1800, 2026).plot(
    kind='hist', bins=60, ax=axes[1], color='mediumpurple', edgecolor='white')
axes[1].set_title('Publication Year Distribution (Raw)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('age_year_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

current_year = datetime.utcnow().year
invalid_ages = users_b[(users_b['age'] < 5) | (users_b['age'] > 100)].shape[0]
invalid_years = books_b[(books_b['year_of_publication'] == 0) |
                        (books_b['year_of_publication'] > current_year)].shape[0]
print(f'Users with invalid age (< 5 or > 100): {invalid_ages:,}')
print(f'Books with invalid year (0 or future) : {invalid_years:,}')

**Finding (Ages & Years):** Many ages are below 5 or above 100 (data entry errors, e.g., 0 and 244). Publication years include 0 and future years. These will be nullified and filled in Silver.

In [ ]:
# ── Ratings per User and per Book ─────────────────────────────────────────
ratings_explicit = ratings_b[ratings_b['book_rating'] > 0].copy()

ratings_per_user = ratings_explicit.groupby('user_id').size()
ratings_per_book = ratings_explicit.groupby('isbn').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ratings_per_user.clip(upper=50).plot(
    kind='hist', bins=50, ax=axes[0], color='dodgerblue', edgecolor='white', log=True)
axes[0].set_title('Ratings per User (clipped at 50)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Ratings')
axes[0].set_ylabel('User Count (log scale)')

ratings_per_book.clip(upper=50).plot(
    kind='hist', bins=50, ax=axes[1], color='tomato', edgecolor='white', log=True)
axes[1].set_title('Ratings per Book (clipped at 50)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Ratings')
axes[1].set_ylabel('Book Count (log scale)')

plt.tight_layout()
plt.savefig('ratings_per_entity.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Users with only 1 explicit rating: {(ratings_per_user == 1).sum():,}')
print(f'Books with only 1 explicit rating: {(ratings_per_book == 1).sum():,}')
print(f'Median ratings per user: {ratings_per_user.median():.0f}')
print(f'Median ratings per book: {ratings_per_book.median():.0f}')

**Finding (Ratings per User/Book):** Highly skewed long-tail distribution — most users and books have only 1–2 ratings. We will apply a minimum threshold of **10 ratings** in Silver to ensure enough signal for collaborative filtering.

In [ ]:
# ── Matrix Sparsity ───────────────────────────────────────────────────────
n_users  = ratings_explicit['user_id'].nunique()
n_books  = ratings_explicit['isbn'].nunique()
n_actual = len(ratings_explicit)
n_possible = n_users * n_books
sparsity = (1 - n_actual / n_possible) * 100

print(f'Unique users     : {n_users:,}')
print(f'Unique books     : {n_books:,}')
print(f'Actual ratings   : {n_actual:,}')
print(f'Possible pairs   : {n_possible:,}')
print(f'Matrix Sparsity  : {sparsity:.4f}%')
print(f'Matrix Density   : {100 - sparsity:.4f}%')

**Finding (Sparsity):** The user-book matrix is extremely sparse (>99.9%). This confirms the need for collaborative filtering with a sparse matrix representation, and the importance of the minimum-ratings threshold to reduce noise.

---
# 🥈 STEP 3: Clean the Data and Build the Silver Layer

In [ ]:
# ── Reload Bronze ─────────────────────────────────────────────────────────
users_s   = pd.read_parquet('bronze/users.parquet').copy()
books_s   = pd.read_parquet('bronze/books.parquet').copy()
ratings_s = pd.read_parquet('bronze/ratings.parquet').copy()

# Convert numeric types
users_s['age']                        = pd.to_numeric(users_s['age'], errors='coerce')
books_s['year_of_publication']        = pd.to_numeric(books_s['year_of_publication'], errors='coerce')
ratings_s['book_rating']              = pd.to_numeric(ratings_s['book_rating'], errors='coerce')
users_s['user_id']                    = pd.to_numeric(users_s['user_id'], errors='coerce')
ratings_s['user_id']                  = pd.to_numeric(ratings_s['user_id'], errors='coerce')

print(f'Loaded → Users: {users_s.shape}, Books: {books_s.shape}, Ratings: {ratings_s.shape}')

In [ ]:
# ── 3a. Remove duplicates ─────────────────────────────────────────────────
u_before, b_before, r_before = len(users_s), len(books_s), len(ratings_s)

users_s.drop_duplicates(inplace=True)
books_s.drop_duplicates(inplace=True)
ratings_s.drop_duplicates(inplace=True)

print(f'Duplicates removed:')
print(f'  Users  : {u_before - len(users_s):,}')
print(f'  Books  : {b_before - len(books_s):,}')
print(f'  Ratings: {r_before - len(ratings_s):,}')

In [ ]:
# ── 3b. Remove 0 ratings, keep only 1–10 explicit ratings ─────────────────
r_before = len(ratings_s)
ratings_s = ratings_s[(ratings_s['book_rating'] >= 1) & (ratings_s['book_rating'] <= 10)]
print(f'Ratings after removing 0s: {r_before:,} → {len(ratings_s):,} (removed {r_before - len(ratings_s):,})')

In [ ]:
# ── 3c. Fix user ages ─────────────────────────────────────────────────────
invalid_mask = (users_s['age'] < 5) | (users_s['age'] > 100)
users_s.loc[invalid_mask, 'age'] = np.nan

median_age = users_s['age'].median()
users_s['age'].fillna(median_age, inplace=True)
users_s['age'] = users_s['age'].round(0).astype(int)

print(f'Median age used for filling: {median_age:.1f}')
print(f'Age range after fix: {users_s["age"].min()} – {users_s["age"].max()}')

In [ ]:
# ── 3d. Split Location into city, state, country ──────────────────────────
def split_location(loc):
    if pd.isna(loc):
        return pd.Series({'city': np.nan, 'state': np.nan, 'country': np.nan})
    parts = [p.strip().lower() for p in str(loc).split(',')]
    city    = parts[0] if len(parts) > 0 else np.nan
    state   = parts[1] if len(parts) > 1 else np.nan
    country = parts[2] if len(parts) > 2 else np.nan
    return pd.Series({'city': city, 'state': state, 'country': country})

loc_parts = users_s['location'].apply(split_location)
users_s = pd.concat([users_s, loc_parts], axis=1)
# Keep original location column for reference, but it's optional
print('Location split sample:')
print(users_s[['location', 'city', 'state', 'country']].head(5).to_string())

In [ ]:
# ── 3e. Fix publication years ─────────────────────────────────────────────
current_year = datetime.utcnow().year
invalid_yr_mask = (books_s['year_of_publication'] == 0) | \
                  (books_s['year_of_publication'] > current_year) | \
                  (books_s['year_of_publication'].isna())
books_s.loc[invalid_yr_mask, 'year_of_publication'] = np.nan

print(f'Books with null publication year after fix: {books_s["year_of_publication"].isna().sum():,}')

In [ ]:
# ── 3f. Clean text columns ────────────────────────────────────────────────
def clean_text(s):
    if pd.isna(s):
        return s
    return str(s).strip().title()

books_s['book_title']  = books_s['book_title'].apply(clean_text)
books_s['book_author'] = books_s['book_author'].apply(clean_text)
books_s['publisher']   = books_s['publisher'].apply(clean_text)
books_s['isbn']        = books_s['isbn'].astype(str).str.strip().str.upper()
ratings_s['isbn']      = ratings_s['isbn'].astype(str).str.strip().str.upper()

# Drop image URL columns (not needed for recommender)
img_cols = [c for c in books_s.columns if 'image' in c.lower()]
books_s.drop(columns=img_cols, inplace=True)
print(f'Dropped image columns: {img_cols}')
print(f'Books columns remaining: {books_s.columns.tolist()}')

In [ ]:
# ── 3g. Keep only ratings whose book and user exist in lookup tables ───────
valid_isbns    = set(books_s['isbn'].dropna().unique())
valid_user_ids = set(users_s['user_id'].dropna().unique())

r_before = len(ratings_s)
ratings_s = ratings_s[
    ratings_s['isbn'].isin(valid_isbns) &
    ratings_s['user_id'].isin(valid_user_ids)
]
print(f'Ratings after referential integrity filter: {r_before:,} → {len(ratings_s):,}')

In [ ]:
# ── 3h. Minimum ratings threshold ─────────────────────────────────────────
# THRESHOLD CHOICE: 10 ratings minimum
# JUSTIFICATION: EDA showed median ratings per user ≈ 1 and median per book ≈ 1.
# Using 10 as threshold removes low-signal entities while retaining enough
# data for meaningful collaborative filtering. Lower thresholds (e.g., 5)
# result in very noisy similarity estimates, while higher thresholds (e.g., 20)
# would reduce coverage too aggressively.
MIN_RATINGS = 10

user_counts = ratings_s['user_id'].value_counts()
book_counts = ratings_s['isbn'].value_counts()

active_users = user_counts[user_counts >= MIN_RATINGS].index
popular_books = book_counts[book_counts >= MIN_RATINGS].index

r_before = len(ratings_s)
ratings_s = ratings_s[
    ratings_s['user_id'].isin(active_users) &
    ratings_s['isbn'].isin(popular_books)
]

# Also filter users and books tables to only those in filtered ratings
users_s = users_s[users_s['user_id'].isin(ratings_s['user_id'].unique())]
books_s = books_s[books_s['isbn'].isin(ratings_s['isbn'].unique())]

print(f'MIN_RATINGS threshold : {MIN_RATINGS}')
print(f'Ratings after threshold: {r_before:,} → {len(ratings_s):,}')
print(f'Retained users : {len(users_s):,}')
print(f'Retained books : {len(books_s):,}')

In [ ]:
# ── Save Silver layer ─────────────────────────────────────────────────────
users_s.to_parquet('silver/users.parquet',   index=False)
books_s.to_parquet('silver/books.parquet',   index=False)
ratings_s.to_parquet('silver/ratings.parquet', index=False)

print('✅ Silver layer saved!')
print(f'   silver/users.parquet   → {users_s.shape}')
print(f'   silver/books.parquet   → {books_s.shape}')
print(f'   silver/ratings.parquet → {ratings_s.shape}')

---
# 🥇 STEP 4: Build the Gold Layer

In [ ]:
# ── Reload Silver ─────────────────────────────────────────────────────────
users_g   = pd.read_parquet('silver/users.parquet')
books_g   = pd.read_parquet('silver/books.parquet')
ratings_g = pd.read_parquet('silver/ratings.parquet')

print(f'Silver reloaded → Users: {users_g.shape}, Books: {books_g.shape}, Ratings: {ratings_g.shape}')

In [ ]:
# ── 4a. Create integer indices for users and books ────────────────────────
# Assign contiguous integer IDs
unique_users = ratings_g['user_id'].unique()
unique_books = ratings_g['isbn'].unique()

user_to_idx = {u: i for i, u in enumerate(unique_users)}
book_to_idx = {b: i for i, b in enumerate(unique_books)}
idx_to_book = {i: b for b, i in book_to_idx.items()}

n_users = len(unique_users)
n_books = len(unique_books)

print(f'Matrix dimensions → {n_users} users × {n_books} books')

In [ ]:
# ── 4b. Build sparse user-book rating matrix ──────────────────────────────
row_indices = ratings_g['user_id'].map(user_to_idx).values
col_indices = ratings_g['isbn'].map(book_to_idx).values
data_values = ratings_g['book_rating'].values.astype(np.float32)

rating_matrix = csr_matrix(
    (data_values, (row_indices, col_indices)),
    shape=(n_users, n_books)
)

matrix_sparsity = 1 - rating_matrix.nnz / (n_users * n_books)
print(f'Sparse matrix shape    : {rating_matrix.shape}')
print(f'Non-zero entries       : {rating_matrix.nnz:,}')
print(f'Matrix sparsity (Gold) : {matrix_sparsity*100:.2f}%')

# Save matrix metadata
import pickle
matrix_data = {
    'rating_matrix': rating_matrix,
    'user_to_idx':   user_to_idx,
    'book_to_idx':   book_to_idx,
    'idx_to_book':   idx_to_book,
    'unique_users':  unique_users,
    'unique_books':  unique_books
}
with open('gold/rating_matrix.pkl', 'wb') as f:
    pickle.dump(matrix_data, f)
print('✅ Sparse rating matrix saved to gold/rating_matrix.pkl')

In [ ]:
# ── 4c. Book lookup table ─────────────────────────────────────────────────
book_lookup = books_g[['isbn', 'book_title', 'book_author']].drop_duplicates(subset='isbn').copy()
book_lookup.to_parquet('gold/book_lookup.parquet', index=False)

print('Book lookup table (sample):')
print(book_lookup.head(5).to_string())
print(f'\n✅ Book lookup saved → {book_lookup.shape}')

In [ ]:
# ── 4d. User-level aggregates ─────────────────────────────────────────────
user_agg = ratings_g.groupby('user_id').agg(
    avg_rating=('book_rating', 'mean'),
    num_ratings=('book_rating', 'count')
).reset_index()
user_agg['avg_rating'] = user_agg['avg_rating'].round(2)
user_agg.to_parquet('gold/user_aggregates.parquet', index=False)

print('User aggregates (sample):')
print(user_agg.head(5).to_string())
print(f'\n✅ User aggregates saved → {user_agg.shape}')

In [ ]:
# ── 4e. Book-level aggregates ─────────────────────────────────────────────
book_agg = ratings_g.groupby('isbn').agg(
    avg_rating=('book_rating', 'mean'),
    num_ratings=('book_rating', 'count')
).reset_index()
book_agg['avg_rating'] = book_agg['avg_rating'].round(2)
book_agg = book_agg.merge(book_lookup, on='isbn', how='left')
book_agg.to_parquet('gold/book_aggregates.parquet', index=False)

print('Book aggregates (sample):')
print(book_agg.head(5).to_string())
print(f'\n✅ Book aggregates saved → {book_agg.shape}')

In [ ]:
# ── Gold layer summary ────────────────────────────────────────────────────
print('=' * 50)
print('  GOLD LAYER SUMMARY')
print('=' * 50)
for f in os.listdir('gold'):
    size = os.path.getsize(f'gold/{f}') / 1024
    print(f'  gold/{f:40s}  {size:8.1f} KB')

---
# 🤖 STEP 5: Build the Recommender (Item-Based Collaborative Filtering)

In [ ]:
# ── 5.1 Load the rating matrix from Gold ─────────────────────────────────
with open('gold/rating_matrix.pkl', 'rb') as f:
    md = pickle.load(f)

rating_matrix = md['rating_matrix']   # shape: (users, books)
user_to_idx   = md['user_to_idx']
book_to_idx   = md['book_to_idx']
idx_to_book   = md['idx_to_book']

book_lookup = pd.read_parquet('gold/book_lookup.parquet')

# Book-centric matrix: each ROW = a book, each COLUMN = a user
book_user_matrix = rating_matrix.T.tocsr()  # shape: (books, users)

print(f'Book-user matrix shape: {book_user_matrix.shape}')
print(f'Books: {book_user_matrix.shape[0]:,}  |  Users: {book_user_matrix.shape[1]:,}')

In [ ]:
# ── 5.2 Compute cosine similarity with NearestNeighbors ───────────────────
print('Fitting NearestNeighbors model (this may take a moment)...')
knn_model = NearestNeighbors(
    n_neighbors=11,       # 10 neighbors + itself
    metric='cosine',
    algorithm='brute',    # brute-force works best for sparse cosine
    n_jobs=-1
)
knn_model.fit(book_user_matrix)
print('✅ KNN model fitted!')

In [ ]:
# ── Helper: ISBN ↔ Title lookups ──────────────────────────────────────────
isbn_to_title  = dict(zip(book_lookup['isbn'], book_lookup['book_title']))
isbn_to_author = dict(zip(book_lookup['isbn'], book_lookup['book_author']))

# Title → ISBN (case-insensitive, partial match)
def title_to_isbn(title_query):
    tq = title_query.strip().lower()
    for isbn, title in isbn_to_title.items():
        if pd.notna(title) and tq in str(title).lower():
            return isbn
    return None

print('Helper functions ready ✅')

In [ ]:
# ── 5.3 get_similar_books(title, n=10) ────────────────────────────────────
def get_similar_books(book_title, n=10, verbose=True):
    """
    Returns top-n books most similar to the given book title
    using cosine similarity via KNN on the book-user rating matrix.
    """
    isbn = title_to_isbn(book_title)
    if isbn is None:
        print(f'❌ Book not found: "{book_title}"')
        return []

    if isbn not in book_to_idx:
        print(f'❌ ISBN {isbn} not in rating matrix.')
        return []

    book_idx   = book_to_idx[isbn]
    book_vector = book_user_matrix[book_idx]

    distances, indices = knn_model.kneighbors(book_vector, n_neighbors=n+1)
    distances = distances.flatten()
    indices   = indices.flatten()

    results = []
    for dist, idx in zip(distances, indices):
        if idx == book_idx:
            continue  # Skip the query book itself
        nbr_isbn  = idx_to_book[idx]
        similarity = 1 - dist
        results.append({
            'isbn':       nbr_isbn,
            'title':      isbn_to_title.get(nbr_isbn, 'Unknown'),
            'author':     isbn_to_author.get(nbr_isbn, 'Unknown'),
            'similarity': round(similarity, 4)
        })

    results = results[:n]

    if verbose:
        print(f'\n📖 Books similar to: "{isbn_to_title.get(isbn, book_title)}"')
        print(f'   Author: {isbn_to_author.get(isbn, "Unknown")}')
        print(f'{'─'*65}')
        for i, r in enumerate(results, 1):
            print(f'  {i:2}. {r["title"][:45]:<46} | Sim={r["similarity"]:.4f}')
            print(f'       by {r["author"]}')

    return results

print('get_similar_books() defined ✅')

In [ ]:
# ── Quick test of get_similar_books ───────────────────────────────────────
# Find a popular book to test with
pop_books = pd.read_parquet('gold/book_aggregates.parquet').sort_values('num_ratings', ascending=False)
test_book_title = pop_books.iloc[0]['book_title']
print(f'Testing with popular book: "{test_book_title}"')
_ = get_similar_books(test_book_title, n=10)

In [ ]:
# ── 5.4 recommend(user_id, n=10) ─────────────────────────────────────────
def recommend(user_id, n=10, min_user_rating=7, verbose=True):
    """
    Item-Based Collaborative Filtering Recommender.

    Algorithm:
    1. Find books the user rated highly (>= min_user_rating).
    2. For each such book, get similar books (via KNN cosine similarity).
    3. Score each candidate by: similarity * user_rating (weighted).
    4. Remove books already read by the user.
    5. Return top-n by aggregated weighted score.

    Parameters
    ----------
    user_id        : int   - The user ID to recommend for.
    n              : int   - Number of recommendations to return.
    min_user_rating: int   - Minimum rating to consider a book 'liked'.
    verbose        : bool  - Print formatted output.

    Returns
    -------
    List of dicts with recommendation details.
    """
    if user_id not in user_to_idx:
        print(f'❌ User {user_id} not found in the matrix.')
        return []

    user_idx    = user_to_idx[user_id]
    user_vector = rating_matrix[user_idx]   # shape: (1, books)

    # Books rated by this user
    rated_indices  = user_vector.nonzero()[1]
    rated_isbns    = {idx_to_book[i] for i in rated_indices}

    # Books liked highly
    liked_books = [
        (idx_to_book[i], user_vector[0, i])
        for i in rated_indices
        if user_vector[0, i] >= min_user_rating
    ]

    if not liked_books:
        # Fallback: take top 5 rated books
        liked_books = sorted(
            [(idx_to_book[i], user_vector[0, i]) for i in rated_indices],
            key=lambda x: x[1], reverse=True
        )[:5]

    # Accumulate similarity scores
    candidate_scores = {}
    for isbn, user_rating in liked_books:
        if isbn not in book_to_idx:
            continue
        book_idx    = book_to_idx[isbn]
        book_vector = book_user_matrix[book_idx]
        distances, indices = knn_model.kneighbors(book_vector, n_neighbors=20)
        for dist, idx in zip(distances.flatten(), indices.flatten()):
            cand_isbn = idx_to_book[idx]
            if cand_isbn in rated_isbns:
                continue   # already read
            sim = 1 - dist
            weighted_score = sim * float(user_rating)
            if cand_isbn not in candidate_scores:
                candidate_scores[cand_isbn] = 0.0
            candidate_scores[cand_isbn] += weighted_score

    # Sort by score
    ranked = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)[:n]

    recommendations = []
    for isbn, score in ranked:
        recommendations.append({
            'isbn':   isbn,
            'title':  isbn_to_title.get(isbn, 'Unknown'),
            'author': isbn_to_author.get(isbn, 'Unknown'),
            'score':  round(score, 4)
        })

    if verbose:
        print(f'\n{'='*70}')
        print(f'  🧑 USER ID: {user_id}  |  Recommendations (top {n})')
        print(f'{'='*70}')

        print(f'\n📚 Books this user rated highly (≥ {min_user_rating}):')
        print(f'{'─'*70}')
        liked_sorted = sorted(liked_books, key=lambda x: x[1], reverse=True)
        for isbn, rating in liked_sorted[:10]:
            title  = isbn_to_title.get(isbn, 'Unknown')
            author = isbn_to_author.get(isbn, 'Unknown')
            print(f'  ★ {rating:.0f}/10  {title[:48]:<50}  by {author}')

        print(f'\n🎯 Recommended Books:')
        print(f'{'─'*70}')
        for i, rec in enumerate(recommendations, 1):
            print(f'  {i:2}. {rec["title"][:48]:<50} | Score={rec["score"]:.4f}')
            print(f'       by {rec["author"]}')

    return recommendations

print('recommend() function defined ✅')

---
# 🎯 STEP 6: Demo and Sanity Check

In [ ]:
# ── Pick 3 real users from the gold ratings ───────────────────────────────
user_agg_gold = pd.read_parquet('gold/user_aggregates.parquet')

# Select users with many ratings for a good demo
top_users = user_agg_gold.sort_values('num_ratings', ascending=False).head(20)
print('Top users by number of ratings:')
print(top_users.to_string(index=False))

demo_user_ids = top_users['user_id'].iloc[:3].tolist()
print(f'\nDemo users selected: {demo_user_ids}')

In [ ]:
# ── DEMO: User 1 ──────────────────────────────────────────────────────────
recs_user1 = recommend(demo_user_ids[0], n=10)

In [ ]:
# ── DEMO: User 2 ──────────────────────────────────────────────────────────
recs_user2 = recommend(demo_user_ids[1], n=10)

In [ ]:
# ── DEMO: User 3 ──────────────────────────────────────────────────────────
recs_user3 = recommend(demo_user_ids[2], n=10)

In [ ]:
# ── Sanity Check: get_similar_books demo ─────────────────────────────────
# Test with second and third most popular books
_ = get_similar_books(pop_books.iloc[1]['book_title'], n=10)
print()
_ = get_similar_books(pop_books.iloc[2]['book_title'], n=10)

In [ ]:
# ── Visualise: Top 10 most-rated books in Gold ────────────────────────────
top10 = pd.read_parquet('gold/book_aggregates.parquet') \
          .sort_values('num_ratings', ascending=False).head(10)

plt.figure(figsize=(12, 6))
bars = plt.barh(
    [t[:40] for t in top10['book_title'].fillna('Unknown')],
    top10['num_ratings'],
    color=plt.cm.viridis(np.linspace(0.2, 0.9, 10))
)
plt.xlabel('Number of Ratings', fontsize=12)
plt.title('Top 10 Most-Rated Books (Gold Layer)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
for bar, val in zip(bars, top10['num_ratings']):
    plt.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('top10_books.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────
print('=' * 65)
print('  PIPELINE SUMMARY')
print('=' * 65)
print(f'  BRONZE  → Users: {u_before:,} | Books: {b_before:,} | Ratings: {r_before:,}')
print(f'  SILVER  → Users: {len(users_s):,} | Books: {len(books_s):,} | Ratings: {len(ratings_s):,}')
print(f'  GOLD    → Matrix: {rating_matrix.shape[0]:,} users × {rating_matrix.shape[1]:,} books')
print(f'            Sparsity: {matrix_sparsity*100:.2f}%')
print()
print('  RECOMMENDER: Item-Based Collaborative Filtering')
print('               Algorithm: KNN with Cosine Similarity (brute-force)')
print('               Scoring  : Similarity × User Rating (weighted sum)')
print('=' * 65)
print('  ✅  Assignment Complete!')
print('  Student : Veda Akash | Roll No: LCI2023057')
print('=' * 65)